# Assignment 2 — Part III continuation

The first ResNet18 job completed the four learning-rate trials and saved both final checkpoints, but Slurm stopped the notebook at the 30-minute limit before the reporting cells were written.

This continuation notebook **does not repeat the four learning-rate trials**. It:

1. reads the saved learning-rate comparison;
2. selects the best learning rate;
3. reruns only the constant-learning-rate and `ReduceLROnPlateau` configurations;
4. saves complete training histories and comparison plots;
5. selects and saves the final ResNet18 checkpoint;
6. writes the final hyperparameter table.

All ResNet18 parameters remain trainable, so this is full fine-tuning.

In [ ]:
from pathlib import Path
import copy
import json
import os
import random
import shutil
import time

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from torchvision.models import resnet18, ResNet18_Weights
from torchvision.transforms import functional as TF
from torchvision.transforms import InterpolationMode

SEED = 42

def set_seed(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)

set_seed()

PROJECT_DIR = Path.cwd()
IMAGE_DIR = PROJECT_DIR / "raw"
TRAIN_SPLIT_CSV = PROJECT_DIR / "train_split.csv"
VAL_SPLIT_CSV = PROJECT_DIR / "validation_split.csv"
LR_RESULTS_CSV = (
    PROJECT_DIR / "results_resnet18" /
    "resnet18_learning_rate_results.csv"
)
RESULTS_DIR = PROJECT_DIR / "results_resnet18"
CHECKPOINT_DIR = PROJECT_DIR / "checkpoints_resnet18"

RESULTS_DIR.mkdir(exist_ok=True)
CHECKPOINT_DIR.mkdir(exist_ok=True)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Device:", device)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

In [ ]:
train_df = pd.read_csv(TRAIN_SPLIT_CSV)
val_df = pd.read_csv(VAL_SPLIT_CSV)
lr_results = pd.read_csv(LR_RESULTS_CSV)

CATEGORY_COLUMN = "garment_types"
IMAGE_COLUMN = "article_id"
LABEL_COLUMN = "label"

class_names = sorted(
    pd.concat([
        train_df[CATEGORY_COLUMN],
        val_df[CATEGORY_COLUMN]
    ]).unique().tolist()
)

class_to_idx = {
    class_name: index
    for index, class_name in enumerate(class_names)
}
NUM_CLASSES = len(class_names)

# Preserve existing label mapping when present.
if LABEL_COLUMN not in train_df.columns:
    train_df[LABEL_COLUMN] = (
        train_df[CATEGORY_COLUMN]
        .map(class_to_idx)
        .astype("int64")
    )

if LABEL_COLUMN not in val_df.columns:
    val_df[LABEL_COLUMN] = (
        val_df[CATEGORY_COLUMN]
        .map(class_to_idx)
        .astype("int64")
    )

BEST_LEARNING_RATE = float(
    lr_results.sort_values(
        ["best_validation_loss",
         "best_validation_accuracy"],
        ascending=[True, False]
    ).iloc[0]["learning_rate"]
)

print("Training images:", len(train_df))
print("Validation images:", len(val_df))
print("Classes:", NUM_CLASSES)
print("Selected LR from completed scan:", BEST_LEARNING_RATE)

display(lr_results)

In [ ]:
class ResizeLongestSideAndPad:
    def __init__(self, target_size=224, fill=0):
        self.target_size = int(target_size)
        self.fill = fill

    def __call__(self, image):
        width, height = image.size
        scale = self.target_size / max(width, height)

        new_width = max(
            1,
            min(round(width * scale), self.target_size)
        )
        new_height = max(
            1,
            min(round(height * scale), self.target_size)
        )

        image = TF.resize(
            image,
            [new_height, new_width],
            interpolation=InterpolationMode.BILINEAR,
            antialias=True
        )

        left = (self.target_size - new_width) // 2
        right = self.target_size - new_width - left
        top = (self.target_size - new_height) // 2
        bottom = self.target_size - new_height - top

        return TF.pad(
            image,
            [left, top, right, bottom],
            fill=self.fill
        )


image_transform = transforms.Compose([
    ResizeLongestSideAndPad(224),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])


class ImageDressDataset(Dataset):
    def __init__(self, dataframe):
        self.data = dataframe.reset_index(drop=True).copy()

    def __len__(self):
        return len(self.data)

    def __getitem__(self, index):
        row = self.data.iloc[index]
        identifier = str(row[IMAGE_COLUMN])
        path = IMAGE_DIR / f"{identifier}.jpg"

        with Image.open(path) as image:
            image = image.convert("RGB")
            image = image_transform(image)

        label = int(row[LABEL_COLUMN])
        return image, label


train_dataset = ImageDressDataset(train_df)
val_dataset = ImageDressDataset(val_df)

NUM_WORKERS = 4
BATCH_SIZE = 32

def make_loaders():
    generator = torch.Generator()
    generator.manual_seed(SEED)

    train_loader = DataLoader(
        train_dataset,
        batch_size=BATCH_SIZE,
        shuffle=True,
        num_workers=NUM_WORKERS,
        pin_memory=True,
        persistent_workers=True,
        generator=generator
    )

    val_loader = DataLoader(
        val_dataset,
        batch_size=BATCH_SIZE,
        shuffle=False,
        num_workers=NUM_WORKERS,
        pin_memory=True,
        persistent_workers=True
    )

    return train_loader, val_loader

In [ ]:
def build_model():
    model = resnet18(weights=ResNet18_Weights.DEFAULT)
    in_features = model.fc.in_features
    model.fc = nn.Linear(in_features, NUM_CLASSES)

    for parameter in model.parameters():
        parameter.requires_grad = True

    assert all(
        parameter.requires_grad
        for parameter in model.parameters()
    )

    return model.to(device)


def train_epoch(model, loader, criterion, optimizer):
    model.train()
    loss_sum = 0.0
    correct = 0
    total = 0

    for images, labels in loader:
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)
        logits = model(images)
        loss = criterion(logits, labels)
        loss.backward()
        optimizer.step()

        batch_size = labels.size(0)
        loss_sum += loss.item() * batch_size
        correct += (
            logits.argmax(1) == labels
        ).sum().item()
        total += batch_size

    return loss_sum / total, correct / total


@torch.inference_mode()
def validate_epoch(model, loader, criterion):
    model.eval()
    loss_sum = 0.0
    correct = 0
    total = 0

    for images, labels in loader:
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        logits = model(images)
        loss = criterion(logits, labels)

        batch_size = labels.size(0)
        loss_sum += loss.item() * batch_size
        correct += (
            logits.argmax(1) == labels
        ).sum().item()
        total += batch_size

    return loss_sum / total, correct / total

In [ ]:
def run_final_experiment(
    name,
    scheduler_name=None,
    max_epochs=8,
    early_stopping_patience=4
):
    set_seed()
    train_loader, val_loader = make_loaders()
    model = build_model()

    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=BEST_LEARNING_RATE
    )

    scheduler = None
    if scheduler_name == "ReduceLROnPlateau":
        scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
            optimizer,
            mode="min",
            factor=0.2,
            patience=2,
            min_lr=1e-7
        )

    checkpoint_path = CHECKPOINT_DIR / f"{name}.pt"

    history = {
        "train_loss": [],
        "train_accuracy": [],
        "validation_loss": [],
        "validation_accuracy": [],
        "learning_rate": []
    }

    best_loss = float("inf")
    best_accuracy = 0.0
    best_epoch = 0
    epochs_without_improvement = 0
    start = time.perf_counter()

    for epoch in range(1, max_epochs + 1):
        current_lr = optimizer.param_groups[0]["lr"]

        train_loss, train_accuracy = train_epoch(
            model, train_loader, criterion, optimizer
        )

        val_loss, val_accuracy = validate_epoch(
            model, val_loader, criterion
        )

        history["train_loss"].append(train_loss)
        history["train_accuracy"].append(train_accuracy)
        history["validation_loss"].append(val_loss)
        history["validation_accuracy"].append(val_accuracy)
        history["learning_rate"].append(current_lr)

        if val_loss < best_loss - 1e-6:
            best_loss = val_loss
            best_accuracy = val_accuracy
            best_epoch = epoch
            epochs_without_improvement = 0

            torch.save({
                "model_state_dict": model.state_dict(),
                "model_name": "resnet18",
                "weights": "ResNet18_Weights.DEFAULT",
                "num_classes": NUM_CLASSES,
                "class_names": class_names,
                "class_to_idx": class_to_idx,
                "learning_rate": BEST_LEARNING_RATE,
                "batch_size": BATCH_SIZE,
                "scheduler": scheduler_name,
                "best_epoch": best_epoch,
                "best_validation_loss": best_loss,
                "best_validation_accuracy": best_accuracy
            }, checkpoint_path)
        else:
            epochs_without_improvement += 1

        if scheduler is not None:
            scheduler.step(val_loss)

        print(
            f"{name} | epoch {epoch:02d}/{max_epochs} | "
            f"lr={current_lr:.2e} | "
            f"train loss={train_loss:.4f} | "
            f"train acc={train_accuracy:.4f} | "
            f"val loss={val_loss:.4f} | "
            f"val acc={val_accuracy:.4f}",
            flush=True
        )

        if epochs_without_improvement >= early_stopping_patience:
            print(
                f"Early stopping: best epoch {best_epoch}",
                flush=True
            )
            break

    elapsed = time.perf_counter() - start

    history_path = RESULTS_DIR / f"{name}_history.json"
    with open(history_path, "w", encoding="utf-8") as file:
        json.dump(history, file, indent=2)

    del model, optimizer, train_loader, val_loader

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return {
        "name": name,
        "scheduler": scheduler_name,
        "learning_rate": BEST_LEARNING_RATE,
        "batch_size": BATCH_SIZE,
        "best_epoch": best_epoch,
        "best_validation_loss": best_loss,
        "best_validation_accuracy": best_accuracy,
        "epochs_completed": len(history["validation_loss"]),
        "elapsed_seconds": elapsed,
        "checkpoint_path": str(checkpoint_path),
        "history": history
    }


constant_run = run_final_experiment(
    name="final_constant_lr_continuation",
    scheduler_name=None
)

scheduler_run = run_final_experiment(
    name="final_reduce_on_plateau_continuation",
    scheduler_name="ReduceLROnPlateau"
)

In [ ]:
comparison = pd.DataFrame([
    {
        "configuration": "Constant learning rate",
        "learning_rate": constant_run["learning_rate"],
        "scheduler": "None",
        "best_epoch": constant_run["best_epoch"],
        "best_validation_loss":
            constant_run["best_validation_loss"],
        "best_validation_accuracy":
            constant_run["best_validation_accuracy"],
        "epochs_completed":
            constant_run["epochs_completed"],
        "elapsed_seconds":
            constant_run["elapsed_seconds"]
    },
    {
        "configuration": "ReduceLROnPlateau",
        "learning_rate": scheduler_run["learning_rate"],
        "scheduler": "ReduceLROnPlateau",
        "best_epoch": scheduler_run["best_epoch"],
        "best_validation_loss":
            scheduler_run["best_validation_loss"],
        "best_validation_accuracy":
            scheduler_run["best_validation_accuracy"],
        "epochs_completed":
            scheduler_run["epochs_completed"],
        "elapsed_seconds":
            scheduler_run["elapsed_seconds"]
    }
]).sort_values(
    ["best_validation_loss",
     "best_validation_accuracy"],
    ascending=[True, False]
).reset_index(drop=True)

comparison.to_csv(
    RESULTS_DIR / "resnet18_scheduler_results.csv",
    index=False
)

display(comparison)

plt.figure(figsize=(10, 6))
for label, run in [
    ("Constant LR", constant_run),
    ("ReduceLROnPlateau", scheduler_run)
]:
    values = run["history"]["validation_loss"]
    plt.plot(
        range(1, len(values) + 1),
        values,
        marker="o",
        label=label
    )

plt.xlabel("Epoch")
plt.ylabel("Validation cross-entropy loss")
plt.title("ResNet18 scheduler comparison: validation loss")
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(
    RESULTS_DIR / "resnet18_scheduler_val_loss.png",
    dpi=200,
    bbox_inches="tight"
)
plt.show()

plt.figure(figsize=(10, 6))
for label, run in [
    ("Constant LR", constant_run),
    ("ReduceLROnPlateau", scheduler_run)
]:
    values = run["history"]["validation_accuracy"]
    plt.plot(
        range(1, len(values) + 1),
        values,
        marker="o",
        label=label
    )

plt.xlabel("Epoch")
plt.ylabel("Validation accuracy")
plt.title("ResNet18 scheduler comparison: validation accuracy")
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(
    RESULTS_DIR / "resnet18_scheduler_val_accuracy.png",
    dpi=200,
    bbox_inches="tight"
)
plt.show()

plt.figure(figsize=(10, 5))
values = scheduler_run["history"]["learning_rate"]
plt.step(
    range(1, len(values) + 1),
    values,
    where="post"
)
plt.xlabel("Epoch")
plt.ylabel("Learning rate")
plt.title("Learning-rate path under ReduceLROnPlateau")
plt.yscale("log")
plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(
    RESULTS_DIR / "resnet18_scheduler_learning_rate.png",
    dpi=200,
    bbox_inches="tight"
)
plt.show()

In [ ]:
best_run = min(
    [constant_run, scheduler_run],
    key=lambda run: (
        run["best_validation_loss"],
        -run["best_validation_accuracy"]
    )
)

FINAL_CHECKPOINT = (
    CHECKPOINT_DIR /
    "final_resnet18_classifier.pt"
)

shutil.copy2(
    best_run["checkpoint_path"],
    FINAL_CHECKPOINT
)

final_hyperparameters = pd.DataFrame({
    "component": [
        "CNN model",
        "Pretraining",
        "Fine-tuning strategy",
        "Original classifier",
        "New classifier",
        "Optimizer",
        "Learning rate",
        "Batch size",
        "Scheduler",
        "Loss function",
        "Early-stopping patience",
        "Best epoch",
        "Best validation loss",
        "Best validation accuracy"
    ],
    "selected_value": [
        "ResNet18",
        "ImageNet (ResNet18_Weights.DEFAULT)",
        "All network parameters trainable",
        "Linear(512, 1000)",
        f"Linear(512, {NUM_CLASSES})",
        "Adam",
        best_run["learning_rate"],
        BATCH_SIZE,
        (
            best_run["scheduler"]
            if best_run["scheduler"] is not None
            else "None"
        ),
        "CrossEntropyLoss",
        4,
        best_run["best_epoch"],
        best_run["best_validation_loss"],
        best_run["best_validation_accuracy"]
    ]
})

final_hyperparameters.to_csv(
    RESULTS_DIR /
    "resnet18_final_hyperparameters.csv",
    index=False
)

display(final_hyperparameters)

print("Final checkpoint:", FINAL_CHECKPOINT)
print("Final scheduler:", best_run["scheduler"])
print("Final validation loss:",
      best_run["best_validation_loss"])
print("Final validation accuracy:",
      best_run["best_validation_accuracy"])

## Interpretation

The completed learning-rate scan selected the learning rate with the lowest validation loss. The constant-learning-rate and scheduler configurations were then compared under the same initialization, split, batch size, and early-stopping rule.

The final checkpoint is selected strictly by validation loss, with validation accuracy used only as a secondary criterion. The test set remains untouched until Part IV.